# PV+BESS Parametric Sizing Study

Sweeps BESS **power** (50-150 MW) and **duration** (1-8 hr) against a fixed
300 MWac PV plant to produce a 5x5 = 25-case sizing matrix.

| Cell | Content |
|------|---------|
| 1 | Configuration (all user-adjustable parameters) |
| 2 | Imports |
| 3 | Fixed PV plant - run once, cache |
| 4 | Parallel sweep (25 PV+BESS cases via ProcessPoolExecutor) |
| 5 | Save results to CSV |
| 6 | 2x2 heatmap grid (IRR, NPV, LCOS, Discharge) |
| 7 | Summary table ranked by IRR |

> **Weather data:** Uses cached NSRDB files for Scurry County, TX
> (`lat=33.0278, lon=-100.0814, year='2017'`) - no API calls required.

In [ ]:
# =============================================================================
# CONFIGURATION - edit this cell to change study parameters
# =============================================================================

# --- Site ---
SITE_LAT = 33.0278
SITE_LON = -100.0814
MET_YEAR = '2017'

# --- Fixed PV plant ---
PV_TARGET_KWAC = 300_000   # 300 MWac
PV_DCAC        = 1.35

# --- BESS sizing sweep ---
POWER_MW_LIST    = [50, 75, 100, 125, 150]    # MW  (17 - 50 % of PV AC)
DURATION_HR_LIST = [1, 2, 4, 6, 8]            # hr

# --- Merchant revenue ---
# Flat placeholder; set PRICE_CSV to a CSV path with column 'da_price_mwh'
# to use real 8760-hour merchant prices.
PPA_RATE                     = 45.0   # $/MWh flat placeholder
PRICE_CSV                    = None   # e.g. '/data/ercot_prices_2022.csv'
CAPACITY_PAYMENT_PER_KW_YEAR = 80.0   # $/kW-yr
ANCILLARY_PER_KW_YEAR        = 15.0   # $/kW-yr

# --- Financial assumptions ---
FIN_KWARGS = dict(
    analysis_period       = 25,
    pv_capex_per_kwdc     = 700.0,    # $/kWdc
    opex_per_kwac_year    = 15.0,     # $/kWac/yr
    ppa_rate              = PPA_RATE,
    ppa_escalation        = 1.0,      # %/yr
    discount_rate         = 8.0,      # % real WACC
    inflation_rate        = 2.5,
    debt_fraction         = 0.0,      # all-equity for clean IRR comparison
    loan_rate             = 5.0,
    loan_term             = 18,
    federal_tax_rate      = 21.0,
    itc_rate              = 0.0,      # no ITC - IRR driven by PPA/capex only
    depreciation_schedule = 'MACRS5',
)

# --- Battery hardware defaults (applied to every case in the sweep) ---
BATT_KWARGS = dict(
    chemistry            = 'LFP',
    capex_per_kwh        = 250.0,     # $/kWh
    capex_per_kw         = 150.0,     # $/kW
    opex_per_kwh_year    = 8.0,       # $/kWh/yr
    soc_min              = 10.0,
    soc_max              = 95.0,
    calendar_degradation = 2.0,       # %/yr linear
)

# --- Compute ---
NUM_WORKERS = 8

# --- Output paths ---
import pathlib
_HERE        = pathlib.Path.cwd()
RESULTS_CSV  = _HERE / 'pv_bess_sizing_study_results.csv'
HEATMAP_PNG  = _HERE / 'pv_bess_sizing_heatmap.png'
TABLE_PNG    = _HERE / 'pv_bess_sizing_table.png'

print(f'Sweep: {len(POWER_MW_LIST)} power levels x {len(DURATION_HR_LIST)} durations'
      f' = {len(POWER_MW_LIST) * len(DURATION_HR_LIST)} cases')
print(f'Output dir: {_HERE}')

In [ ]:
import time
import warnings
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm.notebook import tqdm

from pvsamlab import (
    System, TrackingMode,
    Battery, BessDispatch, PvBessSystem,
    Financial, RevenueStack,
    compute_lcoe, compute_lcos,
)
from pv_bess_sizing_worker import run_bess_case

warnings.filterwarnings('ignore')
sns.set_theme(style='darkgrid', font_scale=0.9)
print('Imports OK')

In [ ]:
# Run the fixed PV plant once to:
#   - download and cache the NSRDB weather file (workers reuse it)
#   - confirm site geometry before the 25-case sweep
print('Building fixed PV plant (300 MWac)...')
pv_plant = System(
    lat=SITE_LAT,
    lon=SITE_LON,
    target_kwac=PV_TARGET_KWAC,
    target_dcac=PV_DCAC,
    met_year=MET_YEAR,
    tracking_mode=TrackingMode.SAT,
)
pv_results = pv_plant.run()

print(f'PV plant:       {pv_plant.kwac:>10,.0f} kWac')
print(f'DC/AC ratio:    {pv_plant.dc_ac_ratio:>10.3f}')
print(f'Annual energy:  {pv_results["annual_energy"]/1e6:>10.1f} GWh/yr')
print(f'Perf ratio:     {pv_results["performance_ratio"]:>10.1f} %')
print()
print('Weather file cached. Workers will reuse it without re-downloading.')

In [ ]:
# --- Load merchant prices ---
if PRICE_CSV is not None:
    _price_df    = pd.read_csv(PRICE_CSV)
    price_series = _price_df.iloc[:, 0].tolist()[:8760]
    print(f'Loaded {len(price_series)} hourly prices from {PRICE_CSV}')
else:
    price_series = [PPA_RATE] * 8760
    print(f'Using flat ${PPA_RATE}/MWh placeholder for all 8760 hours')

revenue_kwargs = dict(
    energy_arbitrage_prices      = price_series,
    capacity_payment_per_kw_year = CAPACITY_PAYMENT_PER_KW_YEAR,
    ancillary_services_per_kw_year = ANCILLARY_PER_KW_YEAR,
)

# --- Build task list ---
tasks = [
    {
        'power_kw':        power_mw * 1000,
        'duration_hr':     duration_hr,
        'lat':             SITE_LAT,
        'lon':             SITE_LON,
        'met_year':        MET_YEAR,
        'pv_target_kwac':  PV_TARGET_KWAC,
        'pv_dcac':         PV_DCAC,
        'financial_kwargs': FIN_KWARGS,
        'batt_kwargs':      BATT_KWARGS,
        'revenue_kwargs':   revenue_kwargs,
    }
    for power_mw   in POWER_MW_LIST
    for duration_hr in DURATION_HR_LIST
]

print(f'\nRunning {len(tasks)} cases with {NUM_WORKERS} parallel workers...')
t_start = time.time()

ok_results  = []
err_results = []

with ProcessPoolExecutor(max_workers=NUM_WORKERS) as executor:
    futures = {executor.submit(run_bess_case, task): task for task in tasks}
    for future in tqdm(as_completed(futures), total=len(futures), desc='BESS cases'):
        res = future.result()
        if res['status'] == 'ok':
            ok_results.append(res)
        else:
            err_results.append(res)

elapsed = time.time() - t_start
print(f'\n{"="*55}')
print(f'Done: {len(ok_results)} OK, {len(err_results)} failed in {elapsed:.1f}s')

if err_results:
    print('\nFailed cases:')
    for e in err_results:
        print(f'  {e["power_mw"]:.0f} MW / {e["duration_hr"]}h: {e.get("error", "unknown")}')

In [ ]:
df = (
    pd.DataFrame(ok_results)
    .sort_values(['power_mw', 'duration_hr'])
    .reset_index(drop=True)
)
df.to_csv(RESULTS_CSV, index=False)
print(f'Saved {len(df)} rows -> {RESULTS_CSV}')
print()

# Quick preview
_preview_cols = [
    'power_mw', 'duration_hr', 'energy_mwh',
    'irr_pct', 'npv_usd', 'lcoe_real_cents_per_kwh',
    'lcos_usd_per_kwh', 'batt_annual_discharge_energy_kwh',
]
df[_preview_cols].head(10)

In [ ]:
def _pivot(df, metric):
    '''Pivot to duration x power matrix; longer duration at top.'''
    return (
        df.pivot(index='duration_hr', columns='power_mw', values=metric)
        .sort_index(ascending=False)
    )

def _best_rc(piv, maximize=True):
    '''Return (row, col) index of best value in pivot.'''
    arr = piv.values
    idx = arr.argmax() if maximize else arr.argmin()
    return np.unravel_index(idx, arr.shape)

def _red_border(ax, row, col):
    '''Draw a red dashed rectangle around cell (row, col).'''
    ax.add_patch(patches.Rectangle(
        (col - 0.5, row - 0.5), 1, 1,
        fill=False, edgecolor='red', linewidth=2.5, linestyle='--',
    ))

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle(
    'PV+BESS Parametric Sizing Study - Scurry County TX\n'
    '300 MWac PV | price_signal dispatch | '
    f'${PPA_RATE:.0f}/MWh + ${CAPACITY_PAYMENT_PER_KW_YEAR:.0f} cap '
    f'+ ${ANCILLARY_PER_KW_YEAR:.0f} anc ($/kW-yr)',
    fontsize=11,
)

# ---- 1. IRR (%) - diverging, center 8% ----
piv = _pivot(df, 'irr_pct')
r, c = _best_rc(piv, maximize=True)
ax = axes[0, 0]
sns.heatmap(piv, ax=ax, annot=True, fmt='.1f', cmap='RdYlGn',
            center=8, linewidths=0.5,
            cbar_kws={'label': 'IRR (%)'})
_red_border(ax, r, c)
ax.set_title('IRR (%)  [diverging, center 8% | red = best]')
ax.set_xlabel('Power (MW)')
ax.set_ylabel('Duration (hr)')

# ---- 2. NPV ($M) - diverging, center 0 ----
piv = _pivot(df, 'npv_usd') / 1e6
r, c = _best_rc(piv, maximize=True)
ax = axes[0, 1]
sns.heatmap(piv, ax=ax, annot=True, fmt='.0f', cmap='RdYlGn',
            center=0, linewidths=0.5,
            cbar_kws={'label': 'NPV ($M)'})
_red_border(ax, r, c)
ax.set_title('NPV ($M)  [diverging, center $0 | red = best]')
ax.set_xlabel('Power (MW)')
ax.set_ylabel('Duration (hr)')

# ---- 3. LCOS ($/kWh) - sequential reversed (lower = better) ----
piv = _pivot(df, 'lcos_usd_per_kwh')
r, c = _best_rc(piv, maximize=False)
ax = axes[1, 0]
sns.heatmap(piv, ax=ax, annot=True, fmt='.3f', cmap='YlOrRd_r',
            linewidths=0.5,
            cbar_kws={'label': 'LCOS ($/kWh)'})
_red_border(ax, r, c)
ax.set_title('LCOS ($/kWh)  [lower is better | red = min]')
ax.set_xlabel('Power (MW)')
ax.set_ylabel('Duration (hr)')

# ---- 4. BESS annual discharge (MWh/yr) ----
piv = _pivot(df, 'batt_annual_discharge_energy_kwh') / 1000
r, c = _best_rc(piv, maximize=True)
ax = axes[1, 1]
sns.heatmap(piv, ax=ax, annot=True, fmt='.0f', cmap='Blues',
            linewidths=0.5,
            cbar_kws={'label': 'Discharge (MWh/yr)'})
_red_border(ax, r, c)
ax.set_title('BESS Annual Discharge (MWh/yr)  [red = max]')
ax.set_xlabel('Power (MW)')
ax.set_ylabel('Duration (hr)')

plt.tight_layout()
plt.savefig(HEATMAP_PNG, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved -> {HEATMAP_PNG}')

In [ ]:
# --- Ranked summary table ---
df_ranked = (
    df.sort_values('irr_pct', ascending=False)
    .reset_index(drop=True)
    .assign(rank=lambda x: x.index + 1)
)

_KEYS = [
    'rank', 'power_mw', 'duration_hr', 'energy_mwh',
    'irr_pct', 'npv_usd', 'lcoe_real_cents_per_kwh',
    'lcos_usd_per_kwh', 'batt_annual_discharge_energy_kwh',
]
_HDRS = [
    'Rank', 'Power MW', 'Dur hr', 'Energy MWh',
    'IRR %', 'NPV $M', 'LCOE c/kWh',
    'LCOS $/kWh', 'Discharge MWh/yr',
]

def _fmt(row):
    return [
        f"{int(row['rank'])}",
        f"{row['power_mw']:.0f}",
        f"{row['duration_hr']:.0f}",
        f"{row['energy_mwh']:.0f}",
        f"{row['irr_pct']:.2f}",
        f"{row['npv_usd'] / 1e6:.1f}",
        f"{row['lcoe_real_cents_per_kwh']:.2f}",
        f"{row['lcos_usd_per_kwh']:.3f}",
        f"{row['batt_annual_discharge_energy_kwh'] / 1000:.0f}",
    ]

cell_text = [_fmt(row) for _, row in df_ranked.iterrows()]
n_rows    = len(df_ranked)

fig, ax = plt.subplots(figsize=(15, max(4.0, 0.45 * n_rows + 1.5)))
ax.axis('off')

tbl = ax.table(
    cellText=cell_text,
    colLabels=_HDRS,
    cellLoc='center',
    loc='center',
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(9)
tbl.auto_set_column_width(col=list(range(len(_HDRS))))

# Header: dark blue / white text
for j in range(len(_HDRS)):
    cell = tbl[0, j]
    cell.set_facecolor('#2c5f8a')
    cell.set_text_props(color='white', fontweight='bold')

# Top-3: light green
for i in range(min(3, n_rows)):
    for j in range(len(_HDRS)):
        tbl[i + 1, j].set_facecolor('#90ee90')

# Alternating background for remaining rows
for i in range(3, n_rows):
    bg = '#f0f0f0' if i % 2 == 0 else 'white'
    for j in range(len(_HDRS)):
        tbl[i + 1, j].set_facecolor(bg)

ax.set_title(
    'PV+BESS Sizing Study - 25 Cases Ranked by IRR  (top 3 in green)',
    fontsize=11, pad=15, fontweight='bold',
)

plt.tight_layout()
plt.savefig(TABLE_PNG, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved -> {TABLE_PNG}')